# Sesión 5 — Herramientas para la presentación de avance (CONECTA, proyecto A: Retención)

**Curso:** Visualización de Datos y Gerencia de la Información  
**Sesión:** 5 de 9  
**Tipo:** Notebook de herramientas

## Para qué sirve

**Este notebook es opcional.** La presentación de avance cuenta la historia que construyeron de la S1 a la S4 y se
arma con `plantilla_avance.pptx`, que trae las instrucciones en sus dos primeras diapositivas. Este notebook sirve si
quieren agregar análisis que respondan otras preguntas del comité (diapositiva 8 de la plantilla: vean «Qué ganan con
lo opcional»), y también para verificar sus cifras de la S4 o guardar gráficos de sesiones anteriores.

**Si solo necesitan traer sus gráficos de la S2 a la S4 a la presentación, vayan a la sección 6: no hace falta
ejecutar nada de este notebook.** Ejecútenlo de arriba abajo; cambien solo
lo que dicen las celdas marcadas **Cambien aquí**. Todo se guarda en
`salidas/`, en PNG y en SVG (el SVG se edita en PowerPoint de Microsoft 365: clic derecho, *Convertir en forma*).

| Sección | Qué produce | Para qué diapositiva de la plantilla |
|---|---|---|
| 1. Preparación | Los datos de la S4 y las cifras de la campaña | — |
| 2. Mapas | Los mapas que el equipo plantee | 8. Análisis opcionales |
| 3. La escala en cifras | La tabla de cada UPZ confirmada frente a su localidad | 8 (la regla de escala) |
| 4. Las cifras y las piezas | Las cifras de la campaña, la ganancia por localidad y la serie mensual | 6. Cuánto vale (para verificar sus cifras) y 8 |
| 5. Cifras de control | Lo que un tablero en Power BI debería reproducir | 8 (si proponen un tablero) |
| 6. Gráficos de sesiones anteriores | Cómo guardarlos | 3, 4, 5 y 7 |

## Qué ganan con lo opcional

Su historia de la S1 a la S4 ya responde dónde está el problema, para quién y cuánto vale. Estos análisis responden
otras preguntas que el comité suele hacer. Elijan los que fortalezcan su historia; ninguno es obligatorio.

| Análisis | Pregunta del comité que responde | Qué gana su presentación | Sección |
|---|---|---|---|
| Mapa de la razón por UPZ | «¿Dónde, exactamente?» | El dónde de un vistazo, sobre la ciudad que el comité conoce, en lugar de una tabla de códigos | 2 |
| El mismo indicador por localidad y por UPZ | «Nosotros operamos por localidad: ¿por qué hablan de UPZ?» | Muestran qué vería la Gerencia a su propia escala y qué cambia al agrupar; con eso proponen a qué escala se decide, se opera y se reporta | 2 y 3 |
| Acercamiento a una localidad | «¿Qué pasa dentro de mi localidad?» | El detalle donde el comité tenga dudas | 2 |
| Serie mensual | «¿Esto es nuevo? ¿Está creciendo?» | La urgencia: si el problema es estable, reciente o creciente, y qué cifra conviene vigilar cada mes | 4 |
| Maqueta de un tablero | «¿Cómo sabremos si la campaña funciona?» | Cómo vigilaría el comité la decisión; es el plano del tablero que construirán en Power BI (S7 y S8). Se arma en PowerPoint con las cifras y las piezas | 4 y 5 |
| Cifras de control | Ninguna: es para ustedes | Comprobar que sus cifras de la S4 coinciden con las del curso antes de exponer | 1 y 5 |

## Guía breve: que el mapa y el tablero no engañen

**Mapas**

1. Pinten tasas o razones, no conteos: un mapa de conteos repite dónde hay más clientes (Wilke, 2019, pp. 172–174).
2. Usen de cuatro a seis clases de color: cada color se reconoce; una escala continua obliga a adivinar el valor
   (Wilke, 2019, pp. 174–175).
3. Marquen las áreas con pocos datos en lugar de pintarlas: unos pocos casos mueven mucho su valor (Wilke, 2019,
   p. 211). La función `mapa(...)` ya raya las UPZ con menos de 80 clientes.
4. La escala cambia la conclusión: los resultados de un análisis por áreas dependen de cómo se agrupen las áreas
   (Openshaw, 1984, pp. 4 y 8). Por eso conviene mirar UPZ y localidad, y decir a qué escala se decide, se opera y se
   reporta, y qué se pierde con cada una.

**Tablero**

5. Un tablero muestra lo más importante, en una pantalla, para vigilarlo de un vistazo; el detalle va en otra página
   (Few, 2004, p. 3).
6. Lo más importante va arriba a la izquierda; lo que se usa después, abajo a la derecha (Knaflic, 2015, pp. 127–138).
7. Cada cifra va con su comparación: una meta, la historia u otro grupo (Few, 2006, p. 5).
8. Si lo que importa es una diferencia, muéstrenla directamente (por ejemplo, la brecha), en lugar de obligar a restar
   (Few, 2006, p. 6).
9. Un color para el mensaje y grises para el resto: si todo destaca, nada destaca (Few, 2006, pp. 12–13).

## Antes de empezar

Descompriman el paquete de la sesión y dejen la carpeta así (el notebook busca `datos/` a su lado):

```
DATAVIZ_sesion_05/
    sesion_05_herramientas.ipynb
    plantilla_avance.pptx
    datos/
        upz_s4.csv              una fila por UPZ, con la cadena de la S4 resuelta
        serie_s4.csv            tasa mensual de pérdida de Hogar: UPZ confirmadas y resto de Bogotá
        conecta_limpio_s3.csv   la tabla limpia de la S3 (clientes activos y serie propia)
        upz.geojson             polígonos de las UPZ
        localidades.geojson     polígonos de las localidades
    salidas/                    se crea sola: aquí quedan los mapas y las piezas
```

Usa las mismas librerías de la S4 (pandas, geopandas y matplotlib). La primera celda de código prepara el estilo de los
gráficos (está plegada: no hay que leerla). La segunda revisa que estén los datos.

## 1. Preparación

In [ ]:
import pathlib
import textwrap

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# El notebook busca datos/ e imagenes/ en su carpeta (o en la de arriba, si está en solucionario/).
BASE = pathlib.Path(".") if pathlib.Path("datos").exists() else pathlib.Path("..")
DATOS, IMAGENES = BASE / "datos", BASE / "imagenes"
IMAGENES.mkdir(exist_ok=True)
# Lo que produce el taller se guarda en salidas/, junto al notebook que se ejecuta (el del estudiante en su carpeta;
# el del solucionario, en solucionario/salidas/): así el solucionario no deja respuestas en la carpeta del estudiante.
SALIDAS = pathlib.Path("salidas")
SALIDAS.mkdir(exist_ok=True)

PALETA = {"principal": "#0F766E", "principal_claro": "#4E9F96", "marino": "#1F2A44", "secundario": "#5B6475",
          "neutro": "#8A93A3", "fondo": "#F3F5F9", "acento": "#B85C0A", "texto": "#1F2937"}
plt.rcParams.update({
    "font.family": "Calibri", "font.size": 11, "axes.edgecolor": PALETA["secundario"],
    "axes.labelcolor": PALETA["texto"], "xtick.color": PALETA["secundario"], "ytick.color": PALETA["secundario"],
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 150, "savefig.dpi": 150,
    "savefig.bbox": "tight"})
# Con Calibri, matplotlib no dibuja bien los tamaños enteros 8 y 9 (se usan 8.5, 9.5 o más) ni el texto a 110 ppp: figure.dpi = 150.


def miles(n, redondeo=0):
    """Entero con espacio de miles; redondeo=-3 redondea a miles."""
    return f"{round(n, redondeo):,.0f}".replace(",", " ")


def decimal(x, d=1):
    return f"{x:.{d}f}".replace(".", ",")


def titulo(fig, texto, subtitulo, y=1.04):
    fig.text(0.01, y, texto, fontsize=15, fontweight="bold", color=PALETA["texto"], ha="left")
    fig.text(0.01, y - 0.055, subtitulo, fontsize=11, color=PALETA["secundario"], ha="left")


def fuente(fig, texto):
    fig.text(0.01, -0.04, texto, fontsize=8.5, color=PALETA["secundario"], ha="left")


def guardar(fig, nombre):
    """Guarda la figura en imagenes/ (la presentación usa estos archivos) y la muestra."""
    fig.savefig(IMAGENES / nombre)
    plt.show()


def limpiar_ejes(ax, x=True, y=False):
    if x:
        ax.set_xticks([]); ax.spines["bottom"].set_visible(False)
    if y:
        ax.set_yticks([])
    ax.spines["left"].set_visible(False); ax.tick_params(axis="y", length=0)

from matplotlib.patches import Patch


def coropleta(ax, gdf, col, cortes, formato, titulo_panel, leyenda, sin_dato=None, cmap="Greens", etiquetas=None):
    """Coropleta con clases (Wilke, 2019, pp. 174–175: de cuatro a seis clases se leen mejor que una escala continua).
    sin_dato: columna booleana; esas áreas se dibujan rayadas, en gris, sin clase."""
    colores = plt.get_cmap(cmap)(np.linspace(0.18, 0.95, len(cortes) - 1))
    clase = pd.cut(gdf[col], cortes, labels=False, include_lowest=True)
    from matplotlib.colors import to_hex
    relleno = [to_hex(colores[int(k)]) if pd.notna(k) else PALETA["fondo"] for k in clase]
    gdf.plot(ax=ax, color=relleno, edgecolor="white", linewidth=0.4)
    etiquetas = etiquetas or [f"{formato(cortes[i])} a {formato(cortes[i + 1])}" for i in range(len(cortes) - 1)]
    manijas = [Patch(facecolor=colores[i], label=etiquetas[i]) for i in range(len(cortes) - 1)]
    if sin_dato is not None and gdf[sin_dato].any():
        gdf[gdf[sin_dato]].plot(ax=ax, facecolor=PALETA["fondo"], edgecolor=PALETA["secundario"], hatch="////",
                                linewidth=0.3)
        manijas.append(Patch(facecolor=PALETA["fondo"], edgecolor=PALETA["secundario"], hatch="////", label="Pocos datos"))
    ax.legend(handles=manijas, title=leyenda, loc="lower left", frameon=True, facecolor="white", edgecolor="none",
              framealpha=0.9, fontsize=9.5, title_fontsize=10)
    ax.set_axis_off()
    ax.set_title(titulo_panel, loc="left", fontsize=11.5)


def rotular(ax, gdf, codigos):
    """Nombres de las UPZ indicadas en una columna a la derecha del mapa, de norte a sur, sin que se crucen."""
    sel = gdf[gdf["codigo_upz"].isin(codigos)].copy()
    sel["punto"] = sel.geometry.representative_point()
    sel = sel.assign(y=sel["punto"].y).sort_values("y", ascending=False)
    x0, y0, x1, y1 = gdf.total_bounds
    nombres = {"ENGATIVA": "Engativá", "FONTIBON": "Fontibón", "USAQUEN": "Usaquén"}
    for k, (_, f) in enumerate(sel.iterrows()):
        yy = y1 - (y1 - y0) * (0.3 + 0.07 * k)
        ax.annotate(f"{f['upz'].title()} ({nombres.get(f['localidad'], f['localidad'].title())})", (f["punto"].x, f["punto"].y),
                    xytext=(x1 + (x1 - x0) * 0.05, yy), textcoords="data", fontsize=9.5, va="center",
                    arrowprops={"arrowstyle": "-", "color": PALETA["secundario"], "linewidth": 0.6})

from matplotlib.patches import FancyBboxPatch, Rectangle

ANCHO, ALTO = 13.33, 7.5          # proporción 16:9 de una pantalla


def pagina(titulo_pagina, idea, pie):
    """Lienzo de una página del tablero: banda de título con la idea central y pie con la fuente."""
    fig = plt.figure(figsize=(ANCHO, ALTO))
    fig.patches.append(Rectangle((0, 0.9), 1, 0.1, transform=fig.transFigure, color=PALETA["marino"], zorder=0))
    fig.text(0.015, 0.955, titulo_pagina, color="white", fontsize=17, fontweight="bold", va="center")
    fig.text(0.015, 0.918, idea, color="white", fontsize=11.5, va="center")
    fig.text(0.015, 0.015, pie, color=PALETA["secundario"], fontsize=8.5)
    return fig


def tarjeta(fig, rect, valor, etiqueta, contexto, color=None):
    """Indicador con contexto (Few, 2006, p. 5): el número, qué es y comparado con qué."""
    ax = fig.add_axes(rect); ax.set_axis_off()
    ax.add_patch(FancyBboxPatch((0, 0), 1, 1, boxstyle="round,pad=0,rounding_size=0.04", transform=ax.transAxes,
                                facecolor=PALETA["fondo"], edgecolor="none"))
    ax.text(0.06, 0.66, valor, fontsize=21, fontweight="bold", color=color or PALETA["principal"], va="center")
    ax.text(0.06, 0.36, etiqueta, fontsize=11, color=PALETA["texto"], va="center")
    ax.text(0.06, 0.15, contexto, fontsize=9.5, color=PALETA["secundario"], va="center")
    return ax


def boton(fig, x, y, texto):
    """Botón de navegación o de filtro de la maqueta (en Power BI será un botón o una segmentación)."""
    ax = fig.add_axes([x, y, 0.1, 0.045]); ax.set_axis_off()
    ax.add_patch(FancyBboxPatch((0, 0), 1, 1, boxstyle="round,pad=0,rounding_size=0.3", transform=ax.transAxes,
                                facecolor="white", edgecolor=PALETA["secundario"]))
    ax.text(0.5, 0.5, texto, ha="center", va="center", fontsize=10, color=PALETA["texto"])

In [ ]:
# Revisa que estén los datos antes de seguir (si falta alguno, dice cuál y dónde ponerlo).
NECESARIOS = ["upz_s4.csv", "serie_s4.csv", "conecta_limpio_s3.csv", "upz.geojson", "localidades.geojson"]
faltan_ = [n for n in NECESARIOS if not (DATOS / n).exists()]
if faltan_:
    raise FileNotFoundError(f"Faltan en {DATOS.resolve()}: {', '.join(faltan_)}. Pongan la carpeta datos/ del paquete "
                            "de la sesión junto a este notebook y vuelvan a ejecutar.")
print(f"Datos completos en {DATOS.resolve()}")

In [ ]:
FUENTE_CONECTA = "Fuente: CONECTA S.A., datos sintéticos del curso (nov. 2025 – oct. 2026). Límites: Datos Abiertos Bogotá."
ESCENARIOS = (0.25, 0.50)
por_upz = pd.read_csv(DATOS / "upz_s4.csv").set_index("codigo_upz")                    # tabla de entrega de la S4
serie = pd.read_csv(DATOS / "serie_s4.csv", dtype={"mes": str}).set_index("mes")      # serie mensual de la S4
d = pd.read_csv(DATOS / "conecta_limpio_s3.csv", dtype={"mes": str})                   # clientes activos (y serie propia)
upz = gpd.read_file(DATOS / "upz.geojson")
localidades = gpd.read_file(DATOS / "localidades.geojson")

# Las UPZ confirmadas son las de la tabla del curso (S4). Un equipo que en la S4 confirmó otras UPZ por una decisión
# propia (por ejemplo, otro umbral) escribe aquí sus códigos: los mapas, las cifras, las piezas y las cifras de control
# siguen sus UPZ, y en la sustentación justifica la decisión. Si fue un error, se deja en None.
MIS_CONFIRMADAS = None      # Cambien aquí solo si confirmaron otras UPZ en la S4; p. ej. [19, 46, 80, 81, 82, 113]
if MIS_CONFIRMADAS is not None:
    faltan = set(MIS_CONFIRMADAS) - set(por_upz.index)
    assert not faltan, f"Estos códigos no están en upz_s4.csv: {sorted(faltan)}"
    por_upz["confirmada"] = por_upz.index.isin(MIS_CONFIRMADAS)
    # La serie de la S4 es la de las UPZ del curso: se recalcula con la misma regla (tasa mensual de pérdida de Hogar).
    h_ = d[d["segmento"] == "Hogar"]
    grupo_ = h_["codigo_upz"].isin(MIS_CONFIRMADAS).map({True: "UPZ confirmadas", False: "Resto de Bogotá"})
    serie = 100 * h_.groupby(["mes", grupo_])["perdida"].mean().unstack()[["UPZ confirmadas", "Resto de Bogotá"]]
    print(f"Atención: el notebook usa las {len(MIS_CONFIRMADAS)} UPZ del equipo, no las de la tabla del curso.")
validas = por_upz[~por_upz["pocos_clientes"]]                           # UPZ con al menos 80 clientes (criterio de la S4)
conf, resto = por_upz[por_upz["confirmada"]], por_upz[~por_upz["confirmada"]]
confirmadas = conf.index                                                 # las UPZ confirmadas (S4, o las del equipo)
NOMBRE = {"ENGATIVA": "Engativá"}                                                      # tildes para los rótulos

# Las dos escalas: por UPZ (la tabla de la S4) y por localidad, sumando pérdidas y esperadas de sus UPZ (no se
# promedian las razones: cada UPZ pesa según sus clientes). Sumapaz (rural) no tiene clientes.
mapa_upz = upz.merge(por_upz.drop(columns=["upz", "localidad"]), left_on="codigo_upz", right_index=True, how="left")
mapa_upz["pocos_clientes"] = mapa_upz["pocos_clientes"].fillna(True).astype(bool)   # UPZ sin clientes: sin dato
por_localidad = por_upz.groupby("localidad")[["clientes", "cliente_mes", "perdidas", "esperadas"]].sum()
por_localidad["tasa_pct"] = 100 * por_localidad["perdidas"] / por_localidad["cliente_mes"]
por_localidad["razon"] = por_localidad["perdidas"] / por_localidad["esperadas"]
mapa_loc = gpd.clip(localidades.merge(por_localidad, left_on="localidad", right_index=True), upz.union_all())

# La tabla de clases del mapa del tablero (S8): siempre con las clases del curso, aunque el equipo cambie las de su mapa.
CORTES = [0, 0.8, 1.0, 1.2, 1.5, np.inf]
ETIQUETAS = ["Menos de 0,8", "0,8 a 1,0", "1,0 a 1,2", "1,2 a 1,5", "1,5 o más"]
clases = por_upz[["upz", "localidad", "clientes", "tasa_pct", "razon", "pocos_clientes", "confirmada"]].copy()
clases["clase_razon"] = pd.cut(clases["razon"], CORTES, labels=ETIQUETAS, include_lowest=True).astype(str)
clases.loc[clases["pocos_clientes"], "clase_razon"] = "Pocos clientes"
clases.reset_index().to_csv(SALIDAS / "upz_clases_s5.csv", index=False, encoding="utf-8")


def arpu_de(t):
    """ARPU de Hogar de un grupo de UPZ: promedio de ingreso_mensual_cop por fila cliente-mes de Hogar (regla del curso)."""
    return (t["arpu_hogar"] * t["cliente_mes_hogar"]).sum() / t["cliente_mes_hogar"].sum()


def ganancia_de(t, f=0.25):
    """Ganancia anual de un grupo: exceso de Hogar × ARPU de Hogar de ese grupo × 12 × f."""
    return t["exceso_hogar"].sum() * arpu_de(t) * 12 * f


# Los indicadores de la campaña (todos de Hogar, a quienes va la campaña), leídos de las tablas de la S4.
hogar = d[d["linea"] == "Hogar"]
activos = hogar.loc[hogar["codigo_upz"].isin(confirmadas) & (hogar["mes"] == d["mes"].max()), "cliente_id"].nunique()
ganancia = {f: ganancia_de(conf, f) for f in ESCENARIOS}
indicadores = {
    "tasa_confirmadas": 100 * conf["perdidas_hogar"].sum() / conf["cliente_mes_hogar"].sum(),
    "tasa_resto": 100 * resto["perdidas_hogar"].sum() / resto["cliente_mes_hogar"].sum(),
    "razon_hogar": conf["perdidas_hogar"].sum() / conf["esperadas_hogar"].sum(),
    "exceso": conf["exceso_hogar"].sum(), "esperadas": conf["esperadas_hogar"].sum(), "arpu": arpu_de(conf),
    "ganancia_25": ganancia[0.25], "ganancia_50": ganancia[0.5],
    "clientes_activos": activos, "costo_maximo_por_cliente": ganancia[0.25] / activos}
# Por localidad, cada una con el ARPU de sus clientes: por eso la suma no da exactamente el total.
ganancia_loc = pd.Series({loc: ganancia_de(g) for loc, g in conf.groupby("localidad")}).sort_values(ascending=False)
print(f"{len(por_upz)} UPZ con clientes; {int(por_upz['pocos_clientes'].sum())} con menos de 80; confirmadas: {len(confirmadas)}")

**Qué calcula la preparación** (todo es de los clientes de **Hogar**, a quienes va la campaña, salvo donde se dice):

| Término | Qué es |
|---|---|
| Razón observada/esperada | Pérdidas de la UPZ sobre las pérdidas esperadas para su mezcla de segmentos (todas las líneas). 1 es lo esperado; 1,5 es 50 % más |
| Tasa mensual de pérdida | Pérdidas sobre filas cliente-mes, en porcentaje |
| Exceso | Clientes perdidos de más en el año: pérdidas observadas menos esperadas |
| ARPU | Lo que paga en promedio un cliente de Hogar al mes (COP) |
| Escenario 1/4 o 1/2 | Supuesto: la fracción del exceso que la campaña recuperaría |
| Ganancia anual | Exceso × ARPU × 12 × la fracción del escenario |
| Costo máximo por cliente | La ganancia del escenario 1/4 repartida entre los clientes de Hogar activos de las UPZ confirmadas |
| Brecha | Diferencia, en puntos porcentuales, entre la tasa de las UPZ confirmadas y la del resto de Bogotá |
| Pocos clientes | UPZ con menos de 80 clientes: su tasa no es confiable y se raya en los mapas |

## 2. Mapas

La función `mapa(...)` dibuja el mapa que el equipo decida: qué se pinta (`"razon"`, `"tasa"` o `"conteo"`), a qué
escala (`"upz"` o `"localidad"`), un acercamiento a una localidad (`zoom="NOMBRE EN MAYÚSCULAS"`), qué UPZ llevan
nombre (`nombres=[códigos]`) y el título de acción, que es obligatorio. Las UPZ con menos de 80 clientes se rayan en
los mapas de razón y de tasa; las confirmadas llevan borde oscuro.

| Variable | Qué pinta | Cuándo sirve |
|---|---|---|
| `"razon"` | Pérdidas observadas sobre esperadas | Para ver dónde se pierde más de lo esperado para la mezcla de clientes |
| `"tasa"` | Tasa mensual de pérdida | Para ver dónde se pierde más por cliente, sin ajustar por la mezcla |
| `"conteo"` | Número de pérdidas en el año | Solo para comparar: sin normalizar, repite dónde hay más clientes |

Cada mapa se guarda en `salidas/` en PNG y en SVG. El SVG se edita en PowerPoint de Microsoft 365 (*Insertar →
Imágenes*, y con clic derecho, *Convertir en forma*); si no tienen esa versión, usen el PNG.

**Nombres de las localidades** para `zoom` (con o sin tildes, en mayúsculas o minúsculas): `ANTONIO NARIÑO`, `BARRIOS UNIDOS`, `BOSA`, `CANDELARIA`, `CHAPINERO`, `CIUDAD BOLIVAR`, `ENGATIVA`, `FONTIBON`, `KENNEDY`, `LOS MARTIRES`, `PUENTE ARANDA`, `RAFAEL URIBE URIBE`, `SAN CRISTOBAL`, `SANTA FE`, `SUBA`, `TEUSAQUILLO`, `TUNJUELITO`, `USAQUEN`, `USME`. Para nombrar otras UPZ
con `nombres=[...]`, `buscar_upz("texto")` muestra su código.

In [ ]:
plt.rcParams["svg.fonttype"] = "none"          # en el SVG, los textos quedan como texto editable
VARIABLES = {
    "razon": {"columna": "razon", "cortes": [0, 0.8, 1.0, 1.2, 1.5, np.inf], "leyenda": "Pérdidas / esperadas",
              "etiquetas": ["Menos de 0,8", "0,8 a 1,0", "1,0 a 1,2", "1,2 a 1,5", "1,5 o más"],
              "que": "Razón entre pérdidas observadas y esperadas, todas las líneas"},
    "tasa": {"columna": "tasa_pct", "cortes": [0, 1.5, 2, 2.5, 3, np.inf], "leyenda": "Pérdida mensual",
             "etiquetas": ["Menos de 1,5 %", "1,5 % a 2 %", "2 % a 2,5 %", "2,5 % a 3 %", "3 % o más"],
             "que": "Tasa mensual de pérdida, todas las líneas"},
    "conteo": {"columna": "perdidas", "leyenda": "Pérdidas en el año", "que": "Número de pérdidas en el año, sin normalizar",
               "cortes": {"upz": [0, 20, 50, 100, 150, np.inf], "localidad": [0, 150, 300, 600, 900, np.inf]},
               "etiquetas": {"upz": ["Menos de 20", "20 a 50", "50 a 100", "100 a 150", "150 o más"],
                             "localidad": ["Menos de 150", "150 a 300", "300 a 600", "600 a 900", "900 o más"]}},
}


def mapa(variable="razon", escala="upz", zoom=None, titulo=None, nombres=None, cortes=None, etiquetas=None,
         colores="Oranges", nombre=None):
    """Mapa del taller con las decisiones del equipo; lo guarda en salidas/<nombre>.png y .svg."""
    assert variable in VARIABLES and escala in ("upz", "localidad"), "variable: razon, tasa o conteo; escala: upz o localidad"
    assert titulo, "Escriban un título de acción: la conclusión que el comité debe leer en el mapa"
    v = VARIABLES[variable]
    cortes = cortes or (v["cortes"][escala] if isinstance(v["cortes"], dict) else v["cortes"])
    etiquetas = etiquetas or (v["etiquetas"][escala] if isinstance(v["etiquetas"], dict) else v["etiquetas"])
    capa = mapa_upz if escala == "upz" else mapa_loc
    limites = gpd.clip(localidades, upz.union_all())
    if zoom:
        zoom = zoom.upper().translate(str.maketrans("ÁÉÍÓÚÜ", "AEIOUU"))      # con o sin tildes (la Ñ se conserva)
        validas_ = sorted(set(capa["localidad"].dropna()))
        assert zoom.upper() in validas_, f"No hay una localidad llamada {zoom}. Las válidas: {', '.join(validas_)}"
        capa = capa[capa["localidad"] == zoom.upper()]
        limites = limites[limites["localidad"] == zoom.upper()]
    rayar = "pocos_clientes" if escala == "upz" and variable != "conteo" else None
    fig, ax = plt.subplots(figsize=(8.5, 9))
    coropleta(ax, capa, v["columna"], cortes, None, "", v["leyenda"], sin_dato=rayar, cmap=colores, etiquetas=etiquetas)
    limites.boundary.plot(ax=ax, color=PALETA["marino"], linewidth=0.6 if escala == "upz" else 1.0)
    if escala == "upz":
        capa[capa["codigo_upz"].isin(confirmadas)].plot(ax=ax, facecolor="none", edgecolor=PALETA["marino"], linewidth=1.6)
        rotular(ax, capa, [n for n in (confirmadas if nombres is None else nombres) if n in set(capa["codigo_upz"])])
    partes = [v["que"] + ",", f"por {'UPZ' if escala == 'upz' else 'localidad'}" + (f" (acercamiento a {zoom.title()})" if zoom else "")]
    detalle_ = "con borde oscuro, las UPZ confirmadas" + ("; rayadas, menos de 80 clientes" if rayar else "") \
        if escala == "upz" else "líneas oscuras: límites de localidad"
    # El título de acción y el subtítulo (el parámetro titulo es el texto; por eso no se usa la función titulo()).
    plt.figtext(0.0, 1.0, titulo, fontsize=16, fontweight="bold", color=PALETA["texto"], ha="left", va="bottom")
    plt.figtext(0.0, 0.975, " ".join(partes) + ". " + detalle_[0].upper() + detalle_[1:], fontsize=11,
                color=PALETA["secundario"], ha="left", va="top")
    fuente(fig, FUENTE_CONECTA)
    archivo = nombre or f"mapa_{variable}_{escala}" + (f"_{zoom.lower()}" if zoom else "")
    fig.savefig(SALIDAS / f"{archivo}.png")
    fig.savefig(SALIDAS / f"{archivo}.svg")
    plt.show()
    print(f"Guardado: salidas/{archivo}.png y salidas/{archivo}.svg")

In [ ]:
def buscar_upz(texto=""):
    """Busca UPZ por parte de su nombre o de su localidad (sin tildes) y muestra su código, para usarlo en nombres=[...]."""
    t = por_upz[["upz", "localidad", "clientes", "razon", "confirmada"]].round(2)
    if texto:
        t = t[t["upz"].str.contains(texto.upper()) | t["localidad"].str.contains(texto.upper())]
    display(t.sort_values(["localidad", "upz"]))


buscar_upz("chapinero")      # cambien el texto por la UPZ o la localidad que busquen; buscar_upz() las muestra todas

In [ ]:
# Cambien aquí: quiten el # de la línea que quieran usar, escriban su título de acción y ejecuten.
# Copien una línea para hacer más mapas; cada uno queda en salidas/ con su nombre.
# mapa("razon", escala="upz", titulo="...su conclusión...", nombre="mapa_upz")
# mapa("razon", escala="localidad", titulo="...su conclusión...", nombre="mapa_localidad")
# mapa("razon", zoom="NOMBRE DE UNA LOCALIDAD", titulo="...su conclusión...", nombre="mapa_acercamiento")

## 3. La escala en cifras

La tabla `escondidas` pone cada UPZ confirmada al lado de su localidad: la razón de la UPZ, la de la localidad y los
clientes de las dos. Sirve para escribir la regla de escala: a qué escala se decide, a qué escala se opera, cómo se
reporta y qué se pierde con cada escala.

In [ ]:
escondidas = por_upz.loc[confirmadas, ["upz", "localidad", "razon"]].rename(columns={"razon": "razon_upz"})
escondidas["razon_localidad"] = escondidas["localidad"].map(por_localidad["razon"])
escondidas["clientes_upz"] = por_upz.loc[confirmadas, "clientes"]
escondidas["clientes_localidad"] = escondidas["localidad"].map(por_localidad["clientes"])
display(escondidas.sort_values("razon_upz", ascending=False).round(2).rename(columns={
    "upz": "UPZ", "localidad": "Localidad", "razon_upz": "Razón de la UPZ", "razon_localidad": "Razón de la localidad",
    "clientes_upz": "Clientes de la UPZ", "clientes_localidad": "Clientes de la localidad"}))
print(f"Razón más alta por localidad: {decimal(por_localidad['razon'].max(), 2)} ({por_localidad['razon'].idxmax().title()})")

## 4. Las cifras y las piezas del tablero

La tabla `cifras` reúne los indicadores de la campaña ya calculados. Ustedes deciden cuáles van en la página de
resumen, con qué se compara cada uno y en qué orden se leen. Las dos piezas (la ganancia por localidad y la serie
mensual) quedan en `salidas/` en PNG y SVG.

**La maqueta del tablero** se arma en PowerPoint con estas cifras y piezas, siguiendo los principios 5 a 9 de la guía
breve: cada indicador como tres cuadros de texto (el valor, qué es y comparado con qué). Si prefieren armarla en
Python, la celda plegada de la preparación ya trae las funciones `pagina` y `tarjeta`:

```python
fig = pagina("La pregunta del comité", "La idea central", "Fuente: ...")
tarjeta(fig, [0.015, 0.735, 0.235, 0.14], "valor", "qué es", "comparado con qué")   # [x, y, ancho, alto]
fig.savefig(SALIDAS / "maqueta_resumen.png")
```

In [ ]:
# Las cifras y las piezas (celda ya hecha).
brecha = serie["UPZ confirmadas"].iloc[-1] - serie["Resto de Bogotá"].iloc[-1]
brecha_inicio = serie["UPZ confirmadas"].iloc[:3].mean() - serie["Resto de Bogotá"].iloc[:3].mean()
trimestre = serie["UPZ confirmadas"].tail(3)
cifras = pd.DataFrame([
    ("Tasa mensual de pérdida de Hogar, UPZ confirmadas", f"{decimal(indicadores['tasa_confirmadas'], 2)} %"),
    ("Tasa mensual de pérdida de Hogar, resto de Bogotá", f"{decimal(indicadores['tasa_resto'], 2)} %"),
    ("Pérdidas de Hogar sobre esperadas, UPZ confirmadas", decimal(indicadores["razon_hogar"], 2)),
    ("Pérdidas esperadas de Hogar en el año, UPZ confirmadas", miles(indicadores["esperadas"])),
    ("Exceso: clientes de Hogar perdidos de más en el año", miles(indicadores["exceso"])),
    ("ARPU mensual de Hogar, UPZ confirmadas (COP)", miles(indicadores["arpu"])),
    ("Ganancia anual si se recupera 1/4 del exceso (COP)", miles(indicadores["ganancia_25"])),
    ("Ganancia anual si se recupera 1/2 del exceso (COP)", miles(indicadores["ganancia_50"])),
    (f"Clientes de Hogar activos en {serie.index[-1]}, UPZ confirmadas", miles(indicadores["clientes_activos"])),
    ("Costo máximo de la campaña por cliente activo, escenario 1/4 (COP)", miles(indicadores["costo_maximo_por_cliente"])),
    (f"Brecha de pérdida de Hogar en {serie.index[-1]} (puntos porcentuales)", decimal(brecha, 2)),
    ("Brecha en los tres primeros meses (puntos porcentuales)", decimal(brecha_inicio, 2)),
], columns=["cifra", "valor"])
cifras.to_csv(SALIDAS / "cifras_campana.csv", index=False, encoding="utf-8")
display(cifras)

loc = ganancia_loc.sort_values()
fig, ax = plt.subplots(figsize=(7, 3.6))
nombres_ = [NOMBRE.get(x, x.title()) for x in loc.index]
ax.barh(nombres_, loc.values / 1e6, color=PALETA["principal"], height=0.55)
for y_, v_ in zip(nombres_, loc.values):
    ax.text(v_ / 1e6 + 1, y_, f"COP {decimal(v_ / 1e6, 1)} M", va="center", fontsize=11)
limpiar_ejes(ax); ax.set_xlim(0, loc.max() / 1e6 * 1.45)
ax.set_title("Ganancia anual por localidad, si se recupera 1/4 del exceso", loc="left", fontsize=13)
fig.text(0.0, -0.04, f"Suman COP {decimal(loc.sum() / 1e6, 1)} M, no {decimal(indicadores['ganancia_25'] / 1e6, 1)} M: "
         "cada localidad usa el ARPU de sus clientes de Hogar.", fontsize=10, color=PALETA["secundario"])
fig.savefig(SALIDAS / "pieza_ganancia_localidad.png"); fig.savefig(SALIDAS / "pieza_ganancia_localidad.svg")
plt.show()

x = np.arange(len(serie))
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(x, serie["UPZ confirmadas"], color=PALETA["acento"], linewidth=2.5, marker="o", markersize=4)
ax.plot(x, serie["Resto de Bogotá"], color=PALETA["neutro"], linewidth=2)
ax.text(x[-1] + 0.2, serie["UPZ confirmadas"].iloc[-1], "UPZ confirmadas", va="center", fontsize=10.5, color=PALETA["acento"])
ax.text(x[-1] + 0.2, serie["Resto de Bogotá"].iloc[-1], "Resto de Bogotá", va="center", fontsize=10.5, color=PALETA["secundario"])
ax.axvspan(len(serie) - 3.5, len(serie) - 0.5, color=PALETA["fondo"], zorder=0)
ax.set_xticks(x, [m_[5:] + "/" + m_[2:4] for m_ in serie.index], fontsize=9.5); ax.set_xlim(-0.5, len(serie) + 1.8)
ax.set_ylim(0, serie.max().max() * 1.2); ax.set_ylabel("Pérdida mensual de Hogar (%)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: decimal(v, 0)))
ax.set_title("Tasa mensual de pérdida de Hogar; sombreado, los últimos tres meses", loc="left", fontsize=12)
fig.savefig(SALIDAS / "pieza_serie.png"); fig.savefig(SALIDAS / "pieza_serie.svg")
plt.show()

## 5. Cifras de control

Las cifras que un tablero deberá reproducir cuando se construya en Power BI (S7 y S8). Sirven si proponen un
tablero en la diapositiva 8; la celda también revisa qué piezas ya están en `salidas/`.

In [ ]:
# Lista de insumos y cifras de control (celda ya hecha): revisa qué piezas hay en salidas/ y da las cifras de control.
insumos = {"Mapas (sección 2)": sorted(p_.name for p_ in SALIDAS.glob("*.svg") if p_.stem not in ("pieza_ganancia_localidad", "pieza_serie")),
           "Cifras de la campaña (sección 4)": (SALIDAS / "cifras_campana.csv").exists(),
           "Ganancia por localidad (sección 4)": (SALIDAS / "pieza_ganancia_localidad.svg").exists(),
           "Serie mensual (sección 4)": (SALIDAS / "pieza_serie.svg").exists()}
for nombre_, estado in insumos.items():
    print(f"{nombre_}: {estado if isinstance(estado, list) else ('listo' if estado else 'FALTA')}")
rc = conf["razon"]
cifras_control = pd.DataFrame([
    {"pagina": "Resumen", "cifras_de_control": f"tasa de Hogar {decimal(indicadores['tasa_confirmadas'], 1)} % (resto "
     f"{decimal(indicadores['tasa_resto'], 1)} %); exceso {miles(indicadores['exceso'])}; ganancia COP "
     f"{decimal(indicadores['ganancia_25'] / 1e6, 1)} M (1/4) y {decimal(indicadores['ganancia_50'] / 1e6, 1)} M (1/2); "
     f"costo máximo COP {miles(indicadores['costo_maximo_por_cliente'], -3)}; suma por localidad COP "
     f"{decimal(ganancia_loc.sum() / 1e6, 1)} M"},
    {"pagina": "Seguimiento", "cifras_de_control": f"brecha de {serie.index[-1]}: {decimal(brecha, 1)} pp; últimos tres "
     f"meses entre {decimal(trimestre.min(), 1)} % y {decimal(trimestre.max(), 1)} %"},
    {"pagina": "Detalle por UPZ (si la incluyen)", "cifras_de_control": f"{len(confirmadas)} confirmadas, razón entre "
     f"{decimal(rc.min(), 2)} y {decimal(rc.max(), 2)}"}])
cifras_control.to_csv(SALIDAS / "cifras_control.csv", index=False, encoding="utf-8")
display(cifras_control)

## 6. Gráficos de sesiones anteriores

Para llevar a la presentación un gráfico de la S2, la S3 o la S4 (diapositivas 3, 4, 5 y 7), abran ese notebook,
ejecútenlo y, en la celda del gráfico, después de dibujarlo, agreguen una línea como esta:

```python
fig.savefig("grafico_principal.png", dpi=200, bbox_inches="tight")
```

El archivo queda en la carpeta de ese notebook. Si la celda no tiene una variable `fig`, usen
`plt.gcf().savefig(...)` antes de `plt.show()`.

## Referencias

Few, S. (2004, 20 de marzo). *Dashboard confusion*. Perceptual Edge. https://www.perceptualedge.com/articles/ie/dashboard_confusion.pdf

Few, S. (2006). *Common pitfalls in dashboard design* [Documento técnico]. Perceptual Edge. https://www.perceptualedge.com/articles/Whitepapers/Common_Pitfalls.pdf

Knaflic, C. N. (2015). *Storytelling with data: A data visualization guide for business professionals*. Wiley.

Openshaw, S. (1984). *The modifiable areal unit problem* (Concepts and Techniques in Modern Geography, n.º 38). Geo Books.

Wilke, C. O. (2019). *Fundamentals of data visualization: A primer on making informative and compelling figures*. O'Reilly Media.